# 🕌 QuranASR v4 — Training (Run All)

**A100 GPU + High RAM → Runtime → Run all → dokunma**

1. Paketler kurulur
2. Ses dosyaları Drive → /tmp'ye kopyalanır (10x hızlı I/O)
3. 5-step smoke test yapılır (her şey çalışıyor mu?)
4. Asıl eğitim başlar (10K step)
5. Test + HuggingFace push

📖 [GitHub](https://github.com/tialkan/QuranASR) · 🤗 [HuggingFace](https://huggingface.co/baristiran) · 📱 [Tilavet](https://tilavet.app)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. SETUP — packages, Drive, GPU check, constants
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'accelerate', 'evaluate',
    'jiwer', 'soundfile', 'librosa', 'huggingface_hub'],
    check=True, capture_output=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], capture_output=True)

from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), '❌ No GPU! Switch to A100 GPU + High RAM.'
print(f'✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB)')

BASE        = '/content/drive/MyDrive/quran_asr'
MODEL_ID    = 'openai/whisper-small'
OUTPUT_DIR  = f'{BASE}/checkpoints/quran-whisper-small'
DATASET_DIR = f'{BASE}/data/hf_dataset'
AUDIO_SRC   = f'{BASE}/data/audio'
AUDIO_LOCAL = '/tmp/quran_audio'
HF_REPO_ID  = 'baristiran/QuranASR'

assert os.path.exists(f'{DATASET_DIR}/train'), '❌ Dataset not found! Run notebook.ipynb Cells 1-6 first.'
print('✅ Setup complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. COPY AUDIO — Drive → /tmp (local SSD, 10x faster I/O)
# rsync is smart: skips existing files, ~5 min for 75K files
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, os

os.makedirs(AUDIO_LOCAL, exist_ok=True)

print('Copying audio to local SSD (first time ~5-10 min, then instant)...')
t0 = __import__('time').time()

result = subprocess.run(
    ['rsync', '-a', '--info=progress2', f'{AUDIO_SRC}/', f'{AUDIO_LOCAL}/'],
    capture_output=True, text=True
)

elapsed = __import__('time').time() - t0

# Count files
n = sum(len(f) for _,_,f in os.walk(AUDIO_LOCAL))
print(f'✅ {n:,} files in /tmp ({elapsed:.0f}s)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. PATCH DATASET — rewrite audio paths from Drive → /tmp
# ══════════════════════════════════════════════════════════════════════════════
from datasets import load_from_disk

train_ds = load_from_disk(f'{DATASET_DIR}/train')
val_ds   = load_from_disk(f'{DATASET_DIR}/val')
print(f'Train:{len(train_ds):,}  Val:{len(val_ds):,}')

def patch_path(example):
    example['audio_path'] = example['audio_path'].replace(AUDIO_SRC, AUDIO_LOCAL)
    return example

train_ds = train_ds.map(patch_path)
val_ds   = val_ds.map(patch_path)

# Verify a sample
sample_path = train_ds[0]['audio_path']
assert os.path.exists(sample_path), f'❌ File not found: {sample_path}'
print(f'✅ Paths patched. Sample: {sample_path}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. BUILD MODEL + COLLATOR + TRAINER
# ══════════════════════════════════════════════════════════════════════════════
import torch, evaluate, glob, threading, time, gc
import soundfile as sf, numpy as np
from dataclasses import dataclass, field
from typing import Any, List, Dict
import transformers
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, TrainerCallback,
)

# Keep-alive
def _alive():
    while True:
        time.sleep(55)
        try:
            from google.colab import output; output.eval_js('void(0)')
        except: pass
threading.Thread(target=_alive, daemon=True).start()

# Model
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='arabic', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False
model.generation_config.language = 'arabic'
model.generation_config.task     = 'transcribe'
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='arabic', task='transcribe')

wer_metric = evaluate.load('wer')

@dataclass
class Collator:
    processor: Any
    decoder_start_token_id: int = 50257
    _err: int = field(default=0, repr=False)

    def __call__(self, features: List[Dict]) -> Dict:
        ins, labs = [], []
        for f in features:
            try:
                a, sr = sf.read(f['audio_path'], dtype='float32', always_2d=False)
                if a.ndim > 1: a = a.mean(axis=1)
                if sr != 16000:
                    import librosa; a = librosa.resample(a, orig_sr=sr, target_sr=16000)
                if len(a) < 1600: raise ValueError('too short')
                feat = self.processor.feature_extractor(a, sampling_rate=16000).input_features[0]
                ids  = self.processor.tokenizer(f['text']).input_ids
            except Exception as e:
                self._err += 1
                if self._err <= 5: print(f'  ⚠️ {f.get("audio_path","?")}: {e}')
                feat = np.zeros((80,3000), dtype='float32')
                ids  = self.processor.tokenizer('').input_ids
            ins.append({'input_features': feat})
            labs.append({'input_ids': ids})
        batch = self.processor.feature_extractor.pad(ins, return_tensors='pt')
        lb = self.processor.tokenizer.pad(labs, return_tensors='pt')
        lbl = lb['input_ids'].masked_fill(lb.attention_mask.ne(1), -100)
        if (lbl[:,0] == self.decoder_start_token_id).all(): lbl = lbl[:,1:]
        batch['labels'] = lbl
        return batch

def compute_metrics(pred):
    p, l = pred.predictions, pred.label_ids
    l[l==-100] = processor.tokenizer.pad_token_id
    ps = processor.tokenizer.batch_decode(p, skip_special_tokens=True)
    ls = processor.tokenizer.batch_decode(l, skip_special_tokens=True)
    return {'wer': round(wer_metric.compute(predictions=ps, references=ls)*100, 4)}

class MemClean(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 200 == 0:
            torch.cuda.empty_cache(); gc.collect()

collator = Collator(processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id)
major = int(transformers.__version__.split('.')[0])

print(f'✅ Model loaded (transformers {transformers.__version__})')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. SMOKE TEST — 5 train steps + 1 eval (catches errors BEFORE wasting GPU)
# ══════════════════════════════════════════════════════════════════════════════
print('🧪 Smoke test: 5 train steps + 1 eval...')
print('   If this fails, fix the error before running full training.\n')

import copy
test_model = copy.deepcopy(model)

test_kw = dict(
    output_dir='/tmp/smoke_test',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1e-5, max_steps=5,
    gradient_checkpointing=True, fp16=True,
    eval_steps=5, save_strategy='no',
    logging_steps=1, report_to=[],
    predict_with_generate=True,
    generation_max_length=225,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)
test_kw['eval_strategy' if major >= 5 else 'evaluation_strategy'] = 'steps'

test_tkw = dict(
    model=test_model, args=Seq2SeqTrainingArguments(**test_kw),
    train_dataset=train_ds.select(range(100)),
    eval_dataset=val_ds.select(range(20)),
    data_collator=collator,
    compute_metrics=compute_metrics,
)
test_tkw['processing_class' if major >= 5 else 'tokenizer'] = processor.feature_extractor

test_trainer = Seq2SeqTrainer(**test_tkw)
test_trainer.train()

del test_model, test_trainer
torch.cuda.empty_cache(); gc.collect()

print('\n✅ Smoke test passed! Training, eval, and metrics all work.')
print('   Starting full training...\n')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. FULL TRAINING — 10K steps, checkpoint to Drive every 500 steps
# ══════════════════════════════════════════════════════════════════════════════
kw = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-5, warmup_steps=500, max_steps=10000,
    gradient_checkpointing=True, fp16=True,
    eval_steps=500, save_strategy='steps', save_steps=500,
    save_total_limit=3, load_best_model_at_end=True,
    metric_for_best_model='wer', greater_is_better=False,
    logging_steps=25, report_to=[],
    predict_with_generate=True, generation_max_length=225,
    dataloader_num_workers=4, dataloader_pin_memory=True,
    push_to_hub=False, remove_unused_columns=False,
)
kw['eval_strategy' if major >= 5 else 'evaluation_strategy'] = 'steps'

tkw = dict(
    model=model, args=Seq2SeqTrainingArguments(**kw),
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5), MemClean()],
)
tkw['processing_class' if major >= 5 else 'tokenizer'] = processor.feature_extractor
trainer = Seq2SeqTrainer(**tkw)

# Resume from checkpoint if exists
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
resume = ckpts[-1] if ckpts else None
print(f'{"🔄 Resuming: "+resume if resume else "🆕 New training run"}')
print(f'Batch:64×1=64 | Eval+Save:every 500 steps | Max:10K\n')

trainer.train(resume_from_checkpoint=resume)

FINAL = f'{OUTPUT_DIR}/final'
trainer.save_model(FINAL)
processor.save_pretrained(FINAL)
print(f'\n✅ TRAINING COMPLETE → {FINAL}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. TEST + PUSH TO HUGGINGFACE
# ══════════════════════════════════════════════════════════════════════════════
import random
from transformers import pipeline
from huggingface_hub import HfApi, login

FINAL = f'{OUTPUT_DIR}/final'

pipe = pipeline('automatic-speech-recognition', model=FINAL, chunk_length_s=30,
                device=0, generate_kwargs={'language':'arabic','task':'transcribe'})

print('Test results:\n')
for i, s in enumerate(random.sample(list(val_ds), 5), 1):
    try:
        r = pipe(s['audio_path'])
        print(f'[{i}] Expected: {s["text"][:90]}')
        print(f'    Model  : {r["text"][:90]}\n')
    except Exception as e:
        print(f'[{i}] ERROR: {e}\n')

# Push
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    print('⚠️ HF_TOKEN not in Colab secrets — skipping push.')
    print('  Add it: Secrets → HF_TOKEN → your token from huggingface.co/settings/tokens')
    raise SystemExit('Set HF_TOKEN and re-run this cell.')

card = f"""---
language: ar
license: mit
tags: [whisper, quran, arabic, speech-recognition, fine-tuned]
base_model: openai/whisper-small
pipeline_tag: automatic-speech-recognition
---
# QuranASR — Whisper Fine-tuned for Quran Recitation

```python
from transformers import pipeline
pipe = pipeline('automatic-speech-recognition', model='{HF_REPO_ID}',
                generate_kwargs={{'language':'arabic','task':'transcribe'}})
print(pipe('quran.mp3')['text'])
```

12 reciters × 6236 verses | batch 64 | lr 1e-5 | A100

[GitHub](https://github.com/tialkan/QuranASR) · [Tilavet App](https://tilavet.app) · [Author](https://huggingface.co/baristiran)
"""
with open(f'{FINAL}/README.md','w') as f: f.write(card)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=FINAL, repo_id=HF_REPO_ID,
                  commit_message='Upload QuranASR whisper-small')
print(f'\n✅ Live at: https://huggingface.co/{HF_REPO_ID}')